# 09 · Stacker with the full-data encoder

The same stacker as notebook 07, with one change: the chunk probabilities of the XLM-R trained on all ~245k texts
(notebook 06, 0.936 alone) are added next to those of the 40k model. The reference is the best stacker of notebook 07
(gradient boosting with all features, 0.940 out-of-fold); all scores are out-of-fold on the same 5 group folds.

In [1]:
# Setup: clone/update the repo on Colab/Kaggle, install requirements, resolve paths, get the data
import os, sys, subprocess
REPO_URL = "https://github.com/anna-kryvulia/ucu-ai-hw-generated-text-detection.git"
BRANCH = "main"   # branch with the code you pushed
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    ROOT = "/content/repo" if "google.colab" in sys.modules else "/kaggle/working/repo"
    if not os.path.exists(ROOT):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/requirements.txt"], check=True)
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, f"{ROOT}/src")

from hwkit import env
from hwkit.download import ensure_data
from hwkit.tracker import Run
P = env.setup(seed=42)
ensure_data(P)

platform: local
    data: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/data
 outputs: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/outputs
   cache: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/outputs/cache
    repo: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection
  device: mps
data ready at /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/data


PosixPath('/Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/data')

In [2]:
# Imports, config; validation and test texts; features of every block for both sets
import json
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from hwkit.config import load_config
from hwkit.split import LABELS, load_split, scores
from hwkit.stack import chunk_features, group_folds, out_of_fold, text_features
from hwkit.submission import make_submission

NOTEBOOK_START = time.time()
cfg = load_config("stack_full")
METRICS = ["macro_f1_testlike", "macro_f1", "accuracy", "f1_human", "f1_fusion", "f1_ai"]

val = load_split(pd.read_csv(P.data / "train.csv", usecols=["id", "text", "label", "domain", "lang"]))
val = val[val["fold"] == 0].reset_index(drop=True)
test = pd.read_csv(P.data / "test.csv")
for df in (val, test):
    df["domain_lang"] = df["domain"] + " | " + df["lang"]
y = val["label"].map({c: i for i, c in enumerate(LABELS)}).to_numpy()
w = val["weight"].to_numpy()
folds = group_folds(val, cfg["cv"]["folds"])
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(val[["domain_lang"]])

def run_features(run_id, split, ids):
    """Chunk features of a saved encoder run, rows in the order of `ids`."""
    pred_dir = P.runs / run_id / "predictions"
    order = pd.read_csv(pred_dir / f"{split}.csv", usecols=["id"])["id"]
    f = chunk_features(dict(np.load(pred_dir / f"chunks_{split}.npz")), len(order), run_id)
    return f.set_index(order).loc[ids].reset_index(drop=True)

def blocks_for(split, df):
    tfidf = pd.read_csv(P.runs / cfg["inputs"]["baseline"] / "predictions" / f"{split}.csv").set_index("id")
    enc = pd.concat([run_features(r, split, df["id"]) for r in cfg["inputs"]["encoders"]], axis=1)
    base_cols = [c for c in enc.columns if "_mean_" in c or "_first_" in c or c.endswith("_log_chunks")]
    return {
        "base": pd.concat([enc[base_cols],
                           np.log(np.clip(tfidf.loc[df["id"], [f"s_{c}" for c in LABELS]], 1e-6, 1))
                           .reset_index(drop=True).add_prefix("tfidf_"),
                           pd.DataFrame(ohe.transform(df[["domain_lang"]]), columns=ohe.categories_[0])], axis=1),
        "shape": enc.drop(columns=base_cols),
        "text": text_features(df["text"]),
    }

t0 = time.time()
B = {"val": blocks_for("val", val), "test": blocks_for("test", test)}
run = Run(cfg["name"], cfg, P)
print({k: v.shape[1] for k, v in B["val"].items()}, f"features; {(time.time() - t0) / 60:.1f} min")

run 'stack_full' at /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/outputs/runs/stack_full  done: nothing yet
{'base': 26, 'shape': 10, 'text': 10} features; 0.1 min


In [3]:
# Does each new feature differ between human and fusion? Median per class on validation (unweighted)
B["val"]["shape"].join(B["val"]["text"]).assign(label=val["label"]).groupby("label").median().T[LABELS].round(3)

label,human,fusion,ai
encoder_xlmr_max_ai,0.001,0.000,1.000
encoder_xlmr_max_fusion,0.227,0.999,0.001
encoder_xlmr_min_human,0.762,0.000,0.000
encoder_xlmr_max_human,0.869,0.001,0.000
encoder_xlmr_std_human,0.006,0.000,0.000
encoder_xlmr_full_max_ai,0.000,0.000,1.000
encoder_xlmr_full_max_fusion,0.103,1.000,0.000
encoder_xlmr_full_min_human,0.893,0.000,0.000
encoder_xlmr_full_max_human,0.980,0.000,0.000
encoder_xlmr_full_std_human,0.004,0.000,0.000


## Variants, each against the reference

In [4]:
# Out-of-fold scores of each feature set and model; every variant is then refitted on all of validation for test
def logreg(scale=True):
    lr = LogisticRegression(C=cfg["logreg_C"], max_iter=5000)
    return make_pipeline(StandardScaler(), lr) if scale else lr

def boosting():
    return HistGradientBoostingClassifier(**cfg["boosting"], early_stopping=False, random_state=42)

def fit_model(make, X, rows):
    m = make()
    key = f"{m.steps[-1][0]}__sample_weight" if hasattr(m, "steps") else "sample_weight"
    return m.fit(X[rows], y[rows], **{key: w[rows]})

VARIANTS = {                                     # name: (feature blocks, model)
    "linear, reference features": (["base"], logreg),
    "gradient boosting, reference features": (["base"], boosting),
    "gradient boosting, all features": (["base", "shape", "text"], boosting),
}
rows, oof, test_prob = [], {}, {}
for name, (blocks, make) in VARIANTS.items():
    t0 = time.time()
    Xv = pd.concat([B["val"][b] for b in blocks], axis=1).to_numpy()
    Xt = pd.concat([B["test"][b] for b in blocks], axis=1).to_numpy()
    oof[name] = out_of_fold(lambda r: fit_model(make, Xv, r), lambda m, r: m.predict_proba(Xv[r]), folds)
    test_prob[name] = fit_model(make, Xv, np.ones(len(val), dtype=bool)).predict_proba(Xt)
    res = scores(val["label"], np.array(LABELS)[oof[name].argmax(axis=1)], val["weight"])
    run.log_metrics(name, res)
    rows.append({"variant": name, **{k: res[k] for k in METRICS}, "minutes": (time.time() - t0) / 60})
ref = json.loads((P.runs / "stack_features" / "metrics.json").read_text())["+ both, gradient boosting"]
summary = pd.DataFrame([{"variant": "reference: notebook 07, 40k encoder only", **{k: ref[k] for k in METRICS}}] + rows)
summary.insert(1, "vs reference", summary["macro_f1_testlike"] - summary.loc[0, "macro_f1_testlike"])
summary.round(4)

,variant,vs reference,macro_f1_testlike,macro_f1,accuracy,f1_human,f1_fusion,f1_ai,minutes
0,"reference: notebook 07, 40k encoder only",0.0000,0.9404,0.9428,0.9520,0.9113,0.9219,0.9951,NaN
1,"linear, reference features",0.0063,0.9467,0.9479,0.9569,0.9156,0.9308,0.9974,0.0235
2,"gradient boosting, reference features",0.0142,0.9546,0.9569,0.9642,0.9307,0.9427,0.9974,1.2047
3,"gradient boosting, all features",0.0238,0.9642,0.9664,0.9720,0.9463,0.9554,0.9975,1.6378


**Findings.** Adding the full-data encoder lifts every variant. The linear stacker reaches 0.947, gradient boosting
on the reference features 0.955, and gradient boosting with chunk shape, seams and style 0.964: +0.024 over the
best stacker of notebook 07 and +0.028 over the full-data encoder alone (0.936). The gains of notebook 07 (non-linear
model +0.008, new features +0.010) add up on top of the better encoder.

In [5]:
# Macro-F1 per domain (over the classes present there): linear stacker vs the best variant
best = summary.iloc[1:].sort_values("macro_f1_testlike").iloc[-1]["variant"]
dom = pd.DataFrame({name: {dl: f1_score(g["label"], np.array(LABELS)[oof[name][g.index].argmax(axis=1)],
                                        labels=sorted(g["label"].unique()), average="macro", zero_division=0)
                           for dl, g in val.groupby("domain_lang")}
                    for name in ("linear, reference features", best)})
dom.columns = ["linear", "best"]
dom["texts"] = val["domain_lang"].value_counts()
dom["change"] = dom["best"] - dom["linear"]
print("best variant:", best)
dom.round(3).sort_values("change")

best variant: gradient boosting, all features


,linear,best,texts,change
gutenberg | de-DE,0.837,0.806,43,-0.031
student_essays | en-EN,0.967,0.973,5411,0.005
blog_authorship_corpus | en-EN,0.905,0.911,4053,0.006
cnn_news | en-EN,0.982,0.991,4429,0.009
house_of_commons | en-EN,0.960,0.975,3186,0.015
bundestag | de-DE,0.947,0.972,5234,0.026
euro_court_cases | en-EN,0.923,0.977,2759,0.054
gutenberg | en-EN,0.791,0.945,692,0.155
arxiv_papers | en-EN,0.750,0.965,1429,0.215


**Findings.** As in notebook 07, gradient boosting helps most where the domain changes the rules: arxiv (+0.22),
gutenberg en (+0.16) and court cases (+0.05). Only gutenberg de gets worse, on 43 texts.

In [6]:
# Save out-of-fold and test probabilities of the best variant; write its submission file
cols = [f"s_{c}" for c in LABELS]
pd.concat([val[["id", "label"]], pd.DataFrame(oof[best], columns=cols)], axis=1).to_csv(run.pred_path("val_oof"), index=False)
pd.concat([test[["id"]], pd.DataFrame(test_prob[best], columns=cols)], axis=1).to_csv(run.pred_path("test"), index=False)
sub = make_submission(test["id"], np.array(LABELS)[test_prob[best].argmax(axis=1)],
                      P.repo / "submissions" / f"{cfg['name']}.csv", P.data)

submission saved: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/submissions/stack_full.csv  (25000 rows)
label
ai        0.512
human     0.259
fusion    0.228


In [7]:
# Distinct warnings logged in this session (should be empty), total time
print(f"total: {(time.time() - NOTEBOOK_START) / 60:.1f} min")
env.warnings_summary(P)

total: 4.0 min


[]